In [16]:
import re
%run monodromyfunctions.ipynb

class Pi1():
    #Established once and for all that each R,L letter applies a dehn twist which effects generators of the fundamental group as follows.
    #For reference, see Baker-Peterson, Character Varieties of Once-Punctured Torus Bundles with Tunnel Number 1 pg. 5-7
    monodromy_dict_R = {97 : 'a', 98 : 'b*a'}
    monodromy_dict_L = {97 : 'a*b', 98 : 'b'}
    
    #This dictionary sends both a and b to 1 and leaves t unchanged.
    abelianize_dict = {97 : '1', 98 : '1'}
    
    def __init__(self, monodromy):
        self.monodromy = monodromy
        self.relator1 = self.make_relator1(monodromy)
        self.relator2 = self.make_relator2(monodromy)
        # self.pi1 = self.make_pi1() 
        self.homology = self.make_homology(monodromy)
      
    #Given a letter R or L applies the appropriate twist.
    def twist(self, twist:str, word:str):
        if twist == 'R':
            return word.translate(self.monodromy_dict_R)
        elif twist == 'L':
            return word.translate(self.monodromy_dict_L)
  
    #Apply the monodromy to the fundamental group to generate two relators. Composition reads right to left.
    #Eliminate a letter from the monodromy word after it's been applied until no letters remain.
    def make_relator1(self, monodromy):
        relator = 'a'
        while monodromy != '':
            relator = self.twist(monodromy[-1], relator)
            monodromy = monodromy[:-1]
        return relator
    
    def make_relator2(self, monodromy):
        relator = 'b'
        while monodromy != '':
            relator = self.twist(monodromy[-1], relator)
            monodromy = monodromy[:-1]
        return relator
    
    #Plug these relators into a group presentation which then gets input to the magma engine.
    # def make_pi1(self, monodromy, relator1, relator2):
    #     return magma(f'FPGroup <a,b,x | x^-1*a*x={relator1}, x^-1*b*x = {relator2}>')
    
    def make_pi1(self):
        F.<a,b,x> = FreeGroup()
        return F / [F(self.relator1)*x^-1*a^-1*x, F(self.relator2)*x^-1*a^-1*x]
    
    
    #Fox-derivative product rule. Don't need to consider inverse functions because only t appears on the lhs with inverse, so we can manually deal with this case.
    
    def product_rule(self, der, word):
        der_indices = [m.start() for m in re.finditer(der, word)]
        
        if der_indices[0] == 0:
            derivative = '1'
        else:
            derivative = word[:der_indices[0]-1]
        
        for i in range(1, len(der_indices)):
            derivative = derivative + ' + ' + word[:der_indices[i]-1]
   
        return derivative
            
    
    def fox_derivative_rel1(self, der):
        if der == 'x':
            return '-x^-1 + x^-1*a'
        elif der == 'l':
            return self.product_rule(der, self.relator1)
        else:
            return f'-x^-1 + {self.product_rule(der, self.relator1)}'
        
    def fox_derivative_rel2(self, der):
        if der == 'x':
            return '-x^-1 + x^-1*l'
        elif der == 'a':
            return self.product_rule(der, self.relator2)
        else:
            return f'-x^-1 + {self.product_rule(der, self.relator2)}'
        
        
    #The integral smith normal form gives a matrix which is reduced via a sequence of row and column operations.
    #The resulting diagonal matrix gives the abelian relation for m and l on the first and second diagonals respectively.
    
    def make_homology(self, monodromy):
        mat = (wordtomatrix(monodromy) - matrix.identity(2)).smith_form()[0]
        self.hom_relator1 = mat[0,0]
        self.hom_relator2 = mat[1,1]
        
        G = AbelianGroup([mat[0,0], mat[1,1], 0])
        
        return G
    
    
#Fix Group to make other functions work. Need to get a sage group in order to generate a representation.
#Use split and a dictionary to replace all elements

In [20]:
Pi1('RLRL').relator1

'a*b*a*b*a*a*b*a'

In [32]:
F.<a,b,x> = AbelianGroup(3)
S = F.algebra(QQ)

In [33]:
S.gens()

(a, b, x)

In [29]:
R.<m,l,t> = PolynomialRing(QQ)

In [34]:
f = S.hom([m,l,t], R)

NotImplementedError: Verification of correctness of homomorphisms from Algebra of Multiplicative Abelian group isomorphic to Z x Z x Z over Rational Field not yet implemented.